# Meridian — Sharpe Head Training (Colab)

Trains the `MeridianS` Sharpe-optimising head on top of the trained return head.
Checkpoints save to Google Drive every 5000 steps. Re-run all cells to resume after a session drop.

**Requires `colab_portfolio_return.ipynb` to have completed first** — `return_head_best.pt` must exist in Drive.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
HF_USERNAME   = 'Verdenroz'
HF_TOKEN      = ''            # HF token with read access (required if repos are private)
                              # create at: huggingface.co/settings/tokens
HF_MODEL_REPO = f'{HF_USERNAME}/meridian-checkpoints'
HF_DATA_REPO  = f'{HF_USERNAME}/meridian-data'

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/meridian'
os.makedirs(f'{DRIVE_DIR}/checkpoints/portfolio', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/mmaps',            exist_ok=True)

# Verify return head checkpoint exists before proceeding
return_best = f'{DRIVE_DIR}/checkpoints/portfolio/return_head_best.pt'
assert os.path.exists(return_best), (
    f'return_head_best.pt not found in Drive.\n'
    f'Complete colab_portfolio_return.ipynb first, or copy the checkpoint to:\n'
    f'  {return_best}'
)
print('Drive ready:', DRIVE_DIR)
print('Return head checkpoint found.')

In [ ]:
# ── 2. HuggingFace login + install deps ───────────────────────────────────────
import subprocess, sys
subprocess.run([
    'pip', 'install', '-q',
    'numpy>=2.1', 'pandas>=2.2', 'pyyaml>=6.0',
    'hmmlearn>=0.3', 'faiss-cpu>=1.8', 'tokenizers>=0.20',
    'huggingface_hub',
], check=True)

from huggingface_hub import login, snapshot_download
if HF_TOKEN:
    login(token=HF_TOKEN)
print('HuggingFace ready.')

In [ ]:
# ── 3. Download training code + symbols file ─────────────────────────────────
import os, shutil
from huggingface_hub import hf_hub_download

if not os.path.exists('/content/meridian'):
    local = snapshot_download(repo_id=HF_MODEL_REPO, allow_patterns='code/**')
    shutil.copytree(os.path.join(local, 'code'), '/content/meridian')
    print('Code downloaded.')
else:
    print('Code already present.')

# portfolio_symbols.txt lives at repo root, not inside code/, so fetch it separately.
symbols_path = '/content/meridian/portfolio_symbols.txt'
if not os.path.exists(symbols_path):
    path = hf_hub_download(repo_id=HF_MODEL_REPO, filename='portfolio_symbols.txt')
    shutil.copy(path, symbols_path)
    print(f'portfolio_symbols.txt downloaded → {symbols_path}')
else:
    print('portfolio_symbols.txt already present.')

In [ ]:
# ── 4. Download data (cached to Drive across sessions) ────────────────────────
# Real shards live in meridian-data; JOINT FDA synth tars are generated locally
# via `make fda-joint` and pushed to meridian-data/fda_joint_tars/ (pulled into
# a dedicated --synth-dir by a later cell).
# Drive FUSE is flaky on dirs with 50k+ files (sporadic EIO on listdir).
import os, shutil, time

def _drive_retry(fn, what, attempts=6, base_sleep=3):
    last = None
    for i in range(attempts):
        try:
            return fn()
        except OSError as e:
            last = e
            print(f'[{what}] Drive EIO (attempt {i + 1}/{attempts}): {e}; sleeping {(i + 1) * base_sleep}s')
            time.sleep((i + 1) * base_sleep)
    raise RuntimeError(
        f'{what} failed after {attempts} retries. Drive FUSE is likely stuck. '
        f'Remount with: from google.colab import drive; drive.flush_and_unmount(); '
        f'drive.mount("/content/drive", force_remount=True); then re-run.'
    ) from last

data_dir  = f'{DRIVE_DIR}/data/mmaps'
train_dir = f'{data_dir}/train'

def _has_data():
    if not os.path.exists(train_dir):
        return False
    entries = _drive_retry(lambda: os.listdir(train_dir), 'list train_dir')
    return any(f.endswith('.f32bin') for f in entries)

has_data = _has_data()

if not has_data:
    snapshot_download(repo_id=HF_DATA_REPO, repo_type='dataset', local_dir=data_dir,
                      allow_patterns=['train/**', 'validation/**', 'test/**'])
    for prefix in [os.path.join(data_dir, 'data', 'mmaps'), os.path.join(data_dir, 'mmaps')]:
        if os.path.exists(os.path.join(prefix, 'train')):
            for item in os.listdir(prefix):
                src = os.path.join(prefix, item)
                dst = os.path.join(data_dir, item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(os.path.join(data_dir, prefix.replace(data_dir + '/', '').split('/')[0]), ignore_errors=True)
            break
    print(f'Data downloaded → {data_dir}')
else:
    print('Data already in Drive.')

In [ ]:
# ── 5. Copy data to local SSD for fast I/O ───────────────────────────────────────────────
# Parallel copy: Drive FUSE serializes shutil.copytree; a thread pool gets
# 8–16× speedup. Each file copy retries on EIO.
import os, shutil, time, concurrent.futures

local_data = '/content/data/mmaps'
drive_data = f'{DRIVE_DIR}/data/mmaps'

def _copy_one(src, dst, attempts=5, base_sleep=1):
    last = None
    for i in range(attempts):
        try:
            if os.path.exists(dst):
                return
            shutil.copy2(src, dst)
            return
        except OSError as e:
            last = e
            time.sleep((i + 1) * base_sleep)
    raise RuntimeError(f'copy failed {src} → {dst}: {last}') from last

def _parallel_copy_dir(src_dir, dst_dir, workers=32):
    os.makedirs(dst_dir, exist_ok=True)
    entries = sorted(os.listdir(src_dir))
    jobs = [(os.path.join(src_dir, f), os.path.join(dst_dir, f)) for f in entries]
    t0 = time.time()
    done = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(_copy_one, s, d) for s, d in jobs]
        for fut in concurrent.futures.as_completed(futures):
            fut.result()
            done += 1
            if done % 2000 == 0 or done == len(jobs):
                rate = done / max(time.time() - t0, 1e-6)
                print(f'    {done}/{len(jobs)} ({rate:.0f} files/s)')
    print(f'  {src_dir} → {dst_dir}: {len(jobs)} files in {time.time() - t0:.1f}s')

if not os.path.exists(f'{local_data}/train'):
    print('Copying data from Drive to local SSD (parallel, one-time per session)...')
    for split in ('train', 'validation', 'test'):
        src_split = os.path.join(drive_data, split)
        if os.path.isdir(src_split):
            _parallel_copy_dir(src_split, os.path.join(local_data, split))
    print('Done.')
else:
    print('Local data already present.')

train_files = [f for f in os.listdir(f'{local_data}/train') if f.endswith('.f32bin')]
real_shards  = sorted(f for f in train_files if not f.startswith('FDA_SYNTH_'))
print(f'Real train shards: {len(real_shards)} (joint FDA synth is fetched into --synth-dir by the next cell)')


In [ ]:
# ── Download + unpack JOINT FDA synth tars into a dedicated --synth-dir ────────
# Joint FDA_SYNTH_*.f32bin windows (100-asset (n_assets,T,F), generated locally
# via `make fda-joint`) are packed into ~100 MiB tars under fda_joint_tars/ in
# the data repo. They go in their OWN dir (NOT the real-shard data-dir): the
# portfolio trainer reads them via --synth-dir, because per-asset and joint synth
# share the FDA_SYNTH_ prefix but have different shapes.
import os, tarfile, shutil
from huggingface_hub import list_repo_files, hf_hub_download

synth_dir   = '/content/data/mmaps_synth_joint'
synth_train = f'{synth_dir}/train'
os.makedirs(synth_train, exist_ok=True)

existing_synth = [f for f in os.listdir(synth_train) if f.startswith('FDA_SYNTH_')]
if existing_synth:
    print(f'Joint FDA synth shards already present: {len(existing_synth)} — skipping download.')
else:
    try:
        tar_files = [f for f in list_repo_files(HF_DATA_REPO, repo_type='dataset')
                     if f.startswith('fda_joint_tars/') and f.endswith('.tar')]
    except Exception as e:
        tar_files = []
        print(f'Could not list {HF_DATA_REPO}: {e}')
    if not tar_files:
        print('No joint FDA tars found — training will run real-data-only. '
              'Generate via `make fda-joint` locally and upload to '
              'meridian-data/fda_joint_tars/.')
    else:
        print(f'Downloading and unpacking {len(tar_files)} joint FDA tar(s) from {HF_DATA_REPO}...')
        extracted = 0
        for tar_name in sorted(tar_files):
            local_tar = hf_hub_download(repo_id=HF_DATA_REPO, filename=tar_name, repo_type='dataset')
            with tarfile.open(local_tar, 'r') as tf:
                for member in tf:
                    if not member.isfile() or not member.name.endswith('.f32bin'):
                        continue
                    dst = os.path.join(synth_train, os.path.basename(member.name))
                    if os.path.exists(dst):
                        continue
                    with tf.extractfile(member) as sf, open(dst, 'wb') as df:
                        shutil.copyfileobj(sf, df)
                    extracted += 1
            os.remove(local_tar)
        print(f'Extracted {extracted} joint FDA synth shards.')

synth_shards = [f for f in os.listdir(synth_train) if f.startswith('FDA_SYNTH_')]
print(f'Joint FDA synth shards in {synth_train}: {len(synth_shards)}')


In [ ]:
# ── 6. Auto-detect GPU ────────────────────────────────────────────────────────
import torch, os
assert torch.cuda.is_available(), 'No GPU — enable it in Runtime → Change runtime type.'

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  ({vram_gb:.0f} GB VRAM)')

# TF32 matmul — free speedup on Ampere+ with negligible numerical impact.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# See colab_portfolio_return.ipynb cell-gpu for the memory model rationale.
PTA_DETERMINISTIC = True
DISABLE_REGIME_BALANCE = False
WARMUP_BATCH_SIZE = 8
USE_COMPILE = True
REGIME_POOL_SIZE = 64

if vram_gb >= 75:
    BATCH_SIZE    = 4
    BPTT_CHUNK    = 16
    PTA_MICRO_BS  = 10
elif vram_gb >= 35:
    BATCH_SIZE    = 4
    BPTT_CHUNK    = 8
    PTA_MICRO_BS  = 4
elif vram_gb >= 22:
    BATCH_SIZE    = 4
    BPTT_CHUNK    = 4
    PTA_MICRO_BS  = 2
else:
    BATCH_SIZE    = 2
    BPTT_CHUNK    = 2
    PTA_MICRO_BS  = 1
    DISABLE_REGIME_BALANCE = True
    print('WARNING: 16GB tier — dropping to B=2 and disabling regime-balanced sampling (spec deviation).')

WORKERS = min(8, os.cpu_count() or 1)
print(
    f'batch={BATCH_SIZE}  warmup_batch={WARMUP_BATCH_SIZE}  bptt_chunk={BPTT_CHUNK}  '
    f'pta_micro_bs={PTA_MICRO_BS}  pta_deterministic={PTA_DETERMINISTIC}  '
    f'regime_pool={REGIME_POOL_SIZE}  compile={USE_COMPILE}  '
    f'regime_balance={not DISABLE_REGIME_BALANCE}  workers={WORKERS}'
)

In [ ]:
# ── 7. Train Sharpe head (auto-resumes from Drive) ────────────────────────────
import os, sys, subprocess

DEBUG = False

CKPT_PORT   = f'{DRIVE_DIR}/checkpoints/portfolio'
return_best = f'{CKPT_PORT}/return_head_best.pt'
sharpe_best = f'{CKPT_PORT}/sharpe_head_best.pt'
warmup_ckpt = f'{CKPT_PORT}/sharpe_cross_asset_warmup.pt'

import re, glob

def _latest_step_ckpt(ckpt_dir, head):
    candidates = []
    for path in glob.glob(os.path.join(ckpt_dir, f'{head}_head_step*.pt')):
        m = re.search(rf'{head}_head_step(\d+)\.pt$', path)
        if m:
            candidates.append((int(m.group(1)), path))
    return max(candidates, key=lambda x: x[0])[1] if candidates else None

# Resume priority: most recent step checkpoint first, then best, then warmup,
# then the trained return head (Sharpe head is trained on top of the return head).
latest_step = _latest_step_ckpt(CKPT_PORT, 'sharpe')
if latest_step is not None:
    resume_from = latest_step
    print(f'Resuming from latest step checkpoint: {resume_from}')
elif os.path.exists(sharpe_best):
    resume_from = sharpe_best
    print(f'Resuming from best-val checkpoint: {resume_from}')
elif os.path.exists(warmup_ckpt):
    resume_from = warmup_ckpt
    print(f'Resuming from cross-asset warmup: {resume_from}')
else:
    resume_from = return_best
    print(f'Cold start from trained return head: {resume_from}')

cmd = [
    sys.executable, '-u',
    '-m', 'train.portfolio_train',
    '--head',                          'sharpe',
    '--data-dir',                      '/content/data/mmaps',
    '--synth-dir',                     '/content/data/mmaps_synth_joint',
    '--symbols-file',                  '/content/meridian/portfolio_symbols.txt',
    '--steps',                         '30000',
    '--batch-size',                    str(BATCH_SIZE),
    '--cross-asset-warmup-batch-size', str(WARMUP_BATCH_SIZE),
    '--seq-len',                       '64',
    '--lr',                            '3e-5',
    '--log-every',                     '10',
    '--eval-every',                    '250',
    '--save-every',                    '250',
    '--bptt-chunk',                    str(BPTT_CHUNK),
    '--pta-micro-batch-size',          str(PTA_MICRO_BS),
    '--regime-pool-size',              str(REGIME_POOL_SIZE),
    '--checkpoint-dir',                CKPT_PORT,
    '--resume-backbone',               resume_from,
    '--workers',                       str(WORKERS),
]
if PTA_DETERMINISTIC:
    cmd.append('--pta-deterministic')
if USE_COMPILE:
    cmd.append('--compile')
if DISABLE_REGIME_BALANCE:
    cmd.append('--disable-regime-balance')
if DEBUG:
    cmd.append('--debug')

env = os.environ.copy()
env['PYTHONPATH']         = '/content/meridian'
env['PYTHONUNBUFFERED']   = '1'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print('Running:', ' '.join(cmd))
print('-' * 80)

process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Training failed with return code {process.returncode}')
print('Training completed.')